In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [6]:
df = pd.read_csv("/content/Titanic-Dataset.csv")

print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Dataset Shape: (891, 12)

Columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [7]:
y = df["Survived"]

In [8]:
X = df.copy()

for col in ['PassengerId']:
    X[col] = X[col].fillna(X[col].median())

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


In [11]:
print('\nMissing values in y_train after re-split:')
print(np.isnan(y_train).sum())


Missing values in y_train after re-split:
0


In [12]:
print('Missing values in X_train:')
display(X_train.isnull().sum())


Missing values in X_train:


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,140
SibSp,0
Parch,0
Ticket,0
Fare,0


In [13]:
print('\nMissing values in y_train:')
# y_train is a numpy array, so we can check for NaN directly
print(np.isnan(y_train).sum())


Missing values in y_train:
0


In [14]:
# Impute 'Age' with its median
X_train['Age'] = X_train['Age'].fillna(X_train['Age'].median())
X_test['Age'] = X_test['Age'].fillna(X_test['Age'].median())

# Impute 'Embarked' with its mode
for dataset in [X_train, X_test]:
    mode_embarked = dataset['Embarked'].mode()[0]
    dataset['Embarked'] = dataset['Embarked'].fillna(mode_embarked)

# The 'Cabin' column has already been dropped in a previous execution.
# Removing the drop operation to avoid KeyError.
# X_train = X_train.drop('Cabin', axis=1)
# X_test = X_test.drop('Cabin', axis=1)

print('Missing values in X_train after imputation:')
display(X_train.isnull().sum())

Missing values in X_train after imputation:


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


In [15]:
scaler = StandardScaler()

# Select only numerical columns for scaling
numeric_cols = X_train.select_dtypes(include=['number']).columns

X_train_scaled = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled = scaler.transform(X_test[numeric_cols])

In [16]:
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()



In [17]:
X_train_bias = np.c_[
    np.ones(X_train_scaled.shape[0]),
    X_train_scaled
]

X_test_bias = np.c_[
    np.ones(X_test_scaled.shape[0]),
    X_test_scaled
]

In [18]:
def gradient_descent(
    X,
    y,
    learning_rate=0.01,
    n_iters=1000
):

    m, n = X.shape

    # Initialize parameters
    theta = np.zeros(n)

    # Store loss values
    losses = []

    for i in range(n_iters):

        # Prediction
        y_pred = X @ theta

        # Error
        error = y_pred - y

        # Mean Squared Error / Cost
        loss = (1 / (2 * m)) * np.sum(error ** 2)

        # Store loss
        losses.append(loss)

        # Gradient
        gradient = (1 / m) * (X.T @ error)

        # Update parameters
        theta = theta - learning_rate * gradient

    return theta, losses

In [19]:
theta, losses = gradient_descent(
    X_train_bias,
    y_train,
    learning_rate=0.0001,
    n_iters=1000
)

In [20]:
print("\n============================================")
print("GRADIENT DESCENT PARAMETERS")
print("============================================")

print("\nIntercept:")
print(theta[0])

print("\nCoefficients:")
print(theta[1:])


GRADIENT DESCENT PARAMETERS

Intercept:
0.03582132658189552

Coefficients:
[ 0.00082668  0.04572808 -0.01385089 -0.00240312 -0.00220657  0.0033518
  0.01042421]


In [22]:
y_pred = X_test_bias @ theta

In [23]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    y_pred
)

In [24]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    y_pred
)

print("\n==========================================")
print("LINEAR REGRESSION USING GRADIENT DESCENT")
print("==========================================")

print("\nMean Absolute Error (MAE):")
print(round(mae, 4))

print("\nMean Squared Error (MSE):")
print(round(mse, 4))

print("\nRoot Mean Squared Error (RMSE):")
print(round(rmse, 4))

print("\nR² Score:")
print(round(r2, 4))


LINEAR REGRESSION USING GRADIENT DESCENT

Mean Absolute Error (MAE):
0.3774

Mean Squared Error (MSE):
0.3302

Root Mean Squared Error (RMSE):
0.5746

R² Score:
-0.3615
